# Satellite Imagery Training

More about CLIP: 
- https://github.com/openai/CLIP

Full work (data prep & analysis & modelling): 
- https://www.kaggle.com/code/bencetar/clip-hard-example-mining-finetuning

In [ ]:
import pandas as pd
import numpy as np
from PIL import Image
import cv2
from tqdm import tqdm
import os
import shutil
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
from transformers import CLIPProcessor, CLIPModel
from datetime import datetime
import yaml
import json
import wandb

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["WANDB_MODE"] = "offline"

In [ ]:
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
CONFIG_PATH = "/kaggle/input/datasets/bencetar/clip-sat-data/train_config.yml"

with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

RUN_ = config.get("run_name", "unknown")
RUN_NAME = f"{RUN_}-{datetime.now().strftime('%y-%m-%d-%H-%M')}"
LOSS_FN = config.get("loss_fn", "ce")
LAYERS = config.get("layers", "clf_head")
EPOCHS = config.get("epochs", 5)
BS = config.get("bs", 64)
GPU = config.get("gpu", False)
LR = config.get("lr", 1e-4)
OPT_ = config.get("optim", "adam")

if OPT_ == 'adam':
    OPTIM = lambda params: torch.optim.Adam(params, lr=LR)
else:
    OPTIM = lambda params: torch.optim.SGD(params, lr=LR)

run = wandb.init(project="CLIP-Sat", name=RUN_NAME, config=config, save_code=True, mode="offline")
device = "cuda" if (GPU and torch.cuda.is_available()) else "cpu"
print(f"Device: {device}")

## Preparation For Training

Short summary of result_df (preprocessed data):
- CLIP-based prediction results (predicted labels, ranks, scores)
- Hard / medium / soft sample difficulty classification.
- Scale score: different scales of satellite imagery.

For more detailed information check the "full work" notebook.

In [ ]:
result_df = pd.read_csv('/kaggle/input/datasets/bencetar/clip-sat-data/result_df.csv', index_col=0)
print("Check result_df first 5 rows:\n", result_df[[c for c in result_df.columns if c != 'image_path']].head())

In [ ]:
# Class name mapping
result_df["class_name"] = result_df["image_path"].apply(lambda p: Path(p).parent.name)
id2class = (
    result_df[["gt_label", "class_name"]]
    .drop_duplicates()
    .sort_values("gt_label")
    .set_index("gt_label")["class_name"]
    .to_dict()
)
class2id = {v:k for k,v in id2class.items()}
print("Class mapping:\n",id2class)

In [ ]:
# Split datasets
train_ratio, test_ratio = 0.3, 0.2

train_df, temp_df = train_test_split(
    result_df,
    test_size=train_ratio,
    stratify=result_df["gt_label"],
    random_state=42
)
valid_df, test_df = train_test_split(
    temp_df,
    test_size=test_ratio,
    stratify=temp_df["gt_label"],
    random_state=42
)

print("Dataset sizes after split:")
print(f"Train size: {len(train_df)}")
print(f"Valid size: {len(valid_df)}")
print(f"Test size: {len(test_df)}")

In [ ]:
# Define class weights
classes = np.unique(train_df["gt_label"])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["gt_label"],
)
alpha = 0.5 # smoothing factor
class_weights = class_weights ** alpha
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

w_by_class = pd.DataFrame({
    "class_id": classes,
    "class_name": [id2class[i] for i in classes],
    "weight": class_weights.cpu().numpy(),
    "count": train_df["gt_label"].value_counts().sort_index()
})
w_by_class = w_by_class.sort_values("count", ascending=False)

print("Checking cls weights and cls counts:\n", w_by_class)

## Create Dataset & Dataloader

In [ ]:
class CLIPTrain(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        label = row["gt_label"]
        image_path = row["image_path"]
        image = Image.open(image_path).convert("RGB")

        return {
            "image": image,
            "label": label,
            "image_path": image_path
        }

In [ ]:
train_dataset = CLIPTrain(train_df)
valid_dataset = CLIPTrain(valid_df)
test_dataset = CLIPTrain(test_df)

captions = [
    f"satellite image of {id2class[i]}"
    for i in range(len(id2class))
]

processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def clip_collate_fn(batch, processor, captions):
    images = [item["image"] for item in batch]
    labels = torch.tensor([item["label"] for item in batch])
    paths = [item["image_path"] for item in batch]

    inputs = processor(
        text=captions,
        images=images,
        return_tensors="pt",
        padding=True
    )

    inputs["labels"] = labels
    inputs["paths"] = paths
    inputs["captions"] = captions
    return inputs

train_loader = DataLoader(
    train_dataset,
    batch_size=BS,
    shuffle=True,
    num_workers=0,
    collate_fn=lambda batch: clip_collate_fn(batch, processor, captions)
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BS,
    shuffle=False,
    num_workers=0,
    collate_fn=lambda batch: clip_collate_fn(batch, processor, captions)
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BS,
    shuffle=False,
    num_workers=0,
    collate_fn=lambda batch: clip_collate_fn(batch, processor, captions)
)

loader = iter(train_loader)
batch = next(loader)
print("Dataloader shape check:")
print(batch["pixel_values"].shape)
print(batch["labels"].shape)

## Model Definitions

In [ ]:
def get_image_features_safe(model, pixel_values):
    """ Fix different transformers input format issues."""
    try:
        feats = model.get_image_features(pixel_values=pixel_values)
    except Exception:
        outputs = model.vision_model(pixel_values=pixel_values)
        feats = outputs.pooler_output

    if not isinstance(feats, torch.Tensor):
        feats = feats.pooler_output

    return feats
    
def build_text_features(model, processor, captions, device):
    model.eval()

    inputs = processor(
        text=captions,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        try:
            text_features = model.get_text_features(**inputs)
        except Exception:
            outputs = model.text_model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"]
            )
            text_features = outputs.pooler_output

        if not isinstance(text_features, torch.Tensor):
            text_features = text_features.pooler_output

        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    return text_features

def clip_zero_shot_inference(dataloader, model, text_features, device):
    model.eval()

    all_preds = []
    all_labels = []
    all_paths = []

    with torch.no_grad():
        for batch in dataloader:

            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)
            paths = batch["paths"]

            image_features = get_image_features_safe(model, pixel_values)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

            logits = image_features @ text_features.T
            preds = logits.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_paths.extend(paths)

    return pd.DataFrame({
        "pred": all_preds,
        "label": all_labels,
        "path": all_paths
    })

def classifier_inference(dataloader, model, device):
    model.eval()

    all_preds = []
    all_labels = []
    all_paths = []

    with torch.no_grad():
        for batch in dataloader:

            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)
            paths = batch["paths"]

            logits = model(pixel_values)
            preds = logits.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_paths.extend(paths)

    return pd.DataFrame({
        "pred": all_preds,
        "label": all_labels,
        "path": all_paths
    })

class FocalLoss(nn.Module):
    def __init__(self, gamma=2, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce_loss)
        loss = ((1 - pt) ** self.gamma) * ce_loss
        return loss.mean()
        
class CLIPClassifier(nn.Module):
    def __init__(self, clip_model, num_classes):
        super().__init__()
        self.clip = clip_model
        self.classifier = nn.Linear(clip_model.config.projection_dim, num_classes)

    def forward(self, pixel_values):
        image_features = get_image_features_safe(self.clip, pixel_values)
        logits = self.classifier(image_features)
        return logits
        
def train_clip_classifier(model, dataloader, optimizer, device, loss_fn='ce', class_weights=None):
    model.train()
    total_loss = 0

    pbar = tqdm(dataloader, desc="Train", leave=False)
    for i, batch in enumerate(pbar, 1):

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        logits = model(pixel_values)

        if loss_fn == 'ce':
            loss = F.cross_entropy(logits, labels, weight=class_weights)
        else:
            focal = FocalLoss(gamma=2, weight=class_weights)
            loss = focal(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

def eval_clip_classifier(model, dataloader, device, loss_fn='ce', class_weights=None):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        pbar = tqdm(dataloader, desc="Val  ", leave=False)
        for i, batch in enumerate(pbar, 1):

            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            logits = model(pixel_values)

            if loss_fn == 'ce':
                loss = F.cross_entropy(logits, labels, weight=class_weights)
            else:
                focal = FocalLoss(gamma=2, weight=class_weights)
                loss = focal(logits, labels)

            total_loss += loss.item()

    return total_loss / len(dataloader)

def training_loop(model, train_loader, valid_loader, optimizer, loss_fn, class_weights, device, epochs=5, model_name="best.pt"):
    best_val_loss = float("inf")

    run.watch(model, log="all", log_freq=10)
    
    for epoch in range(epochs):
        train_loss = train_clip_classifier(model, train_loader, optimizer, device, loss_fn, class_weights)
        val_loss = eval_clip_classifier(model, valid_loader, device, loss_fn, class_weights)
        print(f"Epoch {epoch}: train={train_loss:.4f}, val={val_loss:.4f}")

        run.log({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss
        })
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), model_name)
            artifact = wandb.Artifact(name=model_name, type="model", metadata={"val_loss": best_val_loss})
            artifact.add_file(model_name)
            run.log_artifact(artifact)
    
    print(f"Best validation loss: {best_val_loss:.4f}")


# Define metrics
def accuracy(df, log=False):
    acc = (df["pred"] == df["label"]).mean()
    if log:
        run.log({"test_accuracy": f"{acc:.3f}"})
    return acc

def plot_confusion_matrix(cm, labels, log=False, figsize=(6, 5)):
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(cm, annot=True, fmt='d', cmap=plt.cm.Blues, xticklabels=labels, yticklabels=labels)
    ax.set_title('Confusion Matrix')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

    if log:
        run.log({"test_confusion_matrix": wandb.Image(fig)})
    else:
        plt.show()

    plt.close(fig)

def eval_metrics(df, labels=None, log=False):
    clf_rep = classification_report(df["label"].values, df["pred"].values)
    con_mat = confusion_matrix(df["label"].values, df["pred"].values)
    acc = accuracy(df, log=log)
    print(f"Test set accuracy: {acc:.3f}")
    print(f"Classification report:\n", clf_rep)

    if log:
        run.log({"test_clf_report": wandb.Html(f"<pre>{clf_rep}</pre>")})

    plot_confusion_matrix(con_mat, labels=labels, log=log)
    return clf_rep, con_mat

## Run Experiment

In [ ]:
def run_experiment(train_loader, valid_loader, class_weights, device):
    print(f"\nRunning experiment: {RUN_NAME}")
    model_name = f"{RUN_NAME.lower()}.pt"

    # model
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    model = CLIPClassifier(clip_model, num_classes=len(class2id)).to(device)
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    
    # optimizer
    optimizer = OPTIM(model.parameters())

    # freeze strategy
    if LAYERS == 'clf_head':
        for param in model.clip.parameters():
            param.requires_grad = False

    elif LAYERS == 'top':
        for name, param in model.clip.named_parameters():
            if "vision_model.encoder.layers.11" in name:
                param.requires_grad = True
            else:
                param.requires_grad = False

    # text features
    text_features = build_text_features(clip_model, processor, captions, device)

    # baseline
    baseline_df = clip_zero_shot_inference(valid_loader, clip_model, text_features, device)
    baseline_acc = accuracy(baseline_df)
    print(f"Zero-shot accuracy: {baseline_acc:.3f}")

    # training
    training_loop(model, train_loader, valid_loader, optimizer, LOSS_FN, class_weights, device, EPOCHS, model_name)

    # evaluation
    model.load_state_dict(torch.load(model_name))
    finetuned_df = classifier_inference(valid_loader, model, device)
    finetuned_acc = accuracy(finetuned_df)
    print(f"Finetuned accuracy: {finetuned_acc:.3f}")

    run.log({"baseline_df": baseline_df})
    run.log({"finetuned_df": finetuned_df})
    
    return {
        "experiment": RUN_NAME,
        "baseline_acc": baseline_acc,
        "baseline_df": baseline_df,
        "finetuned_acc": finetuned_acc,
        "finetuned_df": finetuned_df,
    }

start = datetime.now()
result = run_experiment(train_loader, valid_loader, class_weights, device)
print(f"Experiment {result['experiment']} has finished. (Elapsed time: {datetime.now()-start}).")

## Evaluate Model

In [ ]:
def evaluate_models(model_path="best_exp_base.pt", test_loader=None, log=True, device=None):
    pt_files = [str(p) for p in Path('.').rglob('*.pt')]
    assert model_path in pt_files, f"Model path not found: '{model_path}'"

    print(f"Test set results on model: {RUN_NAME}")
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    model = CLIPClassifier(clip_model, num_classes=len(class2id)).to(device)
    model.load_state_dict(torch.load(model_path))
    eval_df = classifier_inference(test_loader, model, device)

    # Calculate metrics
    labs = [id2class[i] for i in range(len(id2class))]
    clf_rep, con_mat = eval_metrics(eval_df, labs, log=log)

    return eval_df

log = True
model_path = f"{RUN_NAME.lower()}.pt"
exp_b_eval_df = evaluate_models(model_path, test_loader, log, device)
run.finish()

### Archiving W&B logs and artifacts

In [ ]:
working_dir = Path("/kaggle/working")
WANDB_BUNDLE_DIR = Path("/kaggle/working/wandb_sync_bundle")
WANDB_BUNDLE_FILES_DIR = WANDB_BUNDLE_DIR / "files"

if WANDB_BUNDLE_DIR.exists():
    shutil.rmtree(WANDB_BUNDLE_DIR)

WANDB_BUNDLE_FILES_DIR.mkdir(parents=True, exist_ok=True)
shutil.copytree(working_dir / "wandb", WANDB_BUNDLE_DIR / "wandb", dirs_exist_ok=True)

bundle_files = []
for path in sorted(working_dir.rglob("*")):
    if not path.is_file():
        continue
    if path.is_relative_to(working_dir / "wandb"):
        continue
    if path.is_relative_to(WANDB_BUNDLE_DIR):
        continue
    if path.name in {"wandb_run.zip", "wandb_sync_bundle.zip"}:
        continue

    relative_path = path.relative_to(working_dir)
    destination = WANDB_BUNDLE_FILES_DIR / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(path, destination)
    bundle_files.append(relative_path.as_posix())

manifest = {
    "source_run_ids": [run.id],
    "run_name": RUN_NAME,
    "model_files": [path for path in bundle_files if path.endswith(".pt")],
    "output_files": bundle_files,
}
(WANDB_BUNDLE_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))

shutil.make_archive(
    "/kaggle/working/wandb_run",
    "zip",
    "/kaggle/working",
    "wandb"
)
shutil.make_archive(
    "/kaggle/working/wandb_sync_bundle",
    "zip",
    "/kaggle/working",
    "wandb_sync_bundle"
)
print("Files in /kaggle/working:")
print(sorted(os.listdir("/kaggle/working")))